# SmoothQuant 量化

**核心思想**：通过按输入通道的等价缩放，把难量化的激活异常值迁移到相对容易量化的权重上，实现更准确的 W8A8 训练后量化（PTQ）。

**算法流程**：

1. 用独立校准数据统计激活每个输入通道的最大绝对值 $A_j$，并统计权重对应行的最大绝对值 $B_j$。
2. 计算 $s_j=A_j^{\alpha}/B_j^{1-\alpha}$，构造 $X'=X/s$、$W'=sW$，满足 $XW=X'W'$。
3. 分别对平滑后的激活和权重执行 INT8 量化，比较 FP32、直接 W8A8 和 SmoothQuant W8A8 输出误差。

**关键优势**：避免异常激活通道撑大量化范围。**边界**：这是 CPU 数值演示，使用反量化后 FP32 matmul，不是原生 INT8 GEMM，也不证明推理加速。

In [ ]:
import torch

torch.manual_seed(42)

def symmetric_int8_qdq(x, dim=None):
    """对称 INT8 [-127,127] 量化-反量化；dim 为保留的维度。"""
    max_abs = x.abs().amax(dim=dim, keepdim=True) if dim is not None else x.abs().max()
    scale = max_abs.clamp_min(1e-8) / 127
    q = torch.clamp(torch.round(x / scale), -127, 127).to(torch.int8)
    return q.float() * scale

def get_smooth_scale(x_calib, weight, alpha=0.5):
    """x_calib:[N,C], weight:[C,D]；按输入通道统计。"""
    assert x_calib.ndim == weight.ndim == 2 and x_calib.shape[1] == weight.shape[0]
    assert 0 <= alpha <= 1
    a = x_calib.abs().amax(dim=0).clamp_min(1e-8)
    b = weight.abs().amax(dim=1).clamp_min(1e-8)
    return a.pow(alpha) / b.pow(1-alpha)

def w8a8_output(x, weight):
    # 激活 per-token（每行）；权重 per-output-channel（每列）
    x_hat = symmetric_int8_qdq(x, dim=1)
    w_hat = symmetric_int8_qdq(weight, dim=0)
    return x_hat @ w_hat

def smoothquant_output(x, weight, s):
    return w8a8_output(x / s, weight * s[:, None])

def metrics(pred, ref):
    delta = (pred - ref).float()
    return {'MAE': delta.abs().mean().item(), 'RMSE': delta.square().mean().sqrt().item()}


## 实验：校准与测试分离

构造少量固定的异常输入通道。只使用 `x_calib` 计算 $s$，随后在独立的 `x_test` 上比较误差。激活采用 per-token 对称 INT8，权重采用 per-output-channel 对称 INT8；这些量化粒度是本 Demo 的选择。

In [ ]:
C, D = 32, 16
x_calib = torch.randn(256, C)
x_test = torch.randn(128, C)
weight = torch.randn(C, D) * 0.2
# 固定异常通道，模拟激活通道幅值不均衡
x_calib[:, 0] *= 80
x_calib[:, 1] *= 30
x_test[:, 0] *= 80
x_test[:, 1] *= 30

fp32 = x_test @ weight
direct = w8a8_output(x_test, weight)
print('Direct W8A8:', metrics(direct, fp32))

for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    s = get_smooth_scale(x_calib, weight, alpha)
    x_s, w_s = x_test / s, weight * s[:, None]
    torch.testing.assert_close(x_s @ w_s, fp32, rtol=1e-4, atol=1e-4)
    print(f'alpha={alpha:.2f}', metrics(smoothquant_output(x_test, weight, s), fp32))

# 零通道的平滑系数保持有限，避免除零
zero_x = torch.zeros(4, C)
assert torch.isfinite(get_smooth_scale(zero_x, weight)).all()
print('FP32 equivalence and zero-channel checks passed')

## 观察与思考

- 比较不同 $\alpha$ 的 MAE/RMSE：平滑不保证在所有输入和参数下都优于直接 W8A8。
- 为什么激活列除以 $s_j$，权重对应行乘以 $s_j$，FP32 输出不变？
- 为什么实际部署通常把激活缩放融合到前置归一化层，并预先变换权重？

**论文**：[SmoothQuant (ICML 2023)](https://proceedings.mlr.press/v202/xiao23c.html)。本 Demo 未实现 LayerNorm 融合、真实 INT8 GEMM、端到端 LLM 精度/性能测试。